# Rising Waters: A Machine Learning Approach to Flood Prediction
### Exploratory Data Analysis, Preprocessing, and Comparative Model Training

This Jupyter Notebook documents the complete analytical workflow for the **Rising Waters** flood prediction system. 

**Objective:** Build a predictive classification model to identify flood risks based on six key meteorological and environmental indicators:
1. `Annual_Rainfall` (mm)
2. `Monsoon_Intensity` (Index 0-10)
3. `Cloud_Coverage` (%)
4. `Humidity` (%)
5. `Temperature` (°C)
6. `River_Discharge` (m³/s)

**Algorithms Tested:**
- Decision Tree Classifier
- Random Forest Classifier
- K-Nearest Neighbors (KNN)
- XGBoost Classifier

## 1. Environment Ingestion & Imports

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, classification_report
)

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

%matplotlib inline
sns.set_theme(style="whitegrid")

## 2. Load Dataset
We read `flood.csv` from the datasets directory.

In [ ]:
dataset_path = os.path.join('..', 'dataset', 'flood.csv')
if not os.path.exists(dataset_path):
    print("[!] Note: Place 'flood.csv' under the 'dataset/' folder to execute this notebook.")
else:
    df = pd.read_csv(dataset_path)
    print(f"[+] Dataset Loaded! Shape: {df.shape[0]} rows, {df.shape[1]} columns.")

In [ ]:
# Display first few rows
if 'df' in locals():
    display(df.head())
else:
    print("Dataset variable not initialized yet.")

## 3. Exploratory Data Analysis (EDA)
Reviewing basic summary metrics and distribution shapes.

In [ ]:
if 'df' in locals():
    print("--- Data Types & Info ---")
    df.info()
    print("\n--- Descriptive Statistics ---")
    display(df.describe())

In [ ]:
# target distribution
if 'df' in locals():
    plt.figure(figsize=(6, 4))
    sns.countplot(x='Flood', data=df, palette='Blues')
    plt.title('Target Variable (Flood) Count Distribution')
    plt.show()

In [ ]:
# Correlation Heatmap
if 'df' in locals():
    plt.figure(figsize=(8, 7))
    sns.heatmap(df.corr(), annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
    plt.title('Feature Correlation Heatmap')
    plt.show()

## 4. Preprocessing & Outlier Filtering
We handle missing values, duplicates, and filter outliers using the Interquartile Range (IQR) method.

In [ ]:
if 'df' in locals():
    # 1. Missing Values
    nulls = df.isnull().sum().sum()
    print(f"[*] Total missing values: {nulls}")
    if nulls > 0:
        for col in df.columns:
            df[col] = df[col].fillna(df[col].median())
            
    # 2. Duplicates
    dups = df.duplicated().sum()
    print(f"[*] Total duplicate records: {dups}")
    if dups > 0:
        df = df.drop_duplicates().reset_index(drop=True)
        
    # 3. IQR Outlier Removal
    features = ['Annual_Rainfall', 'Monsoon_Intensity', 'Cloud_Coverage', 'Humidity', 'Temperature', 'River_Discharge']
    orig_rows = df.shape[0]
    outlier_mask = pd.Series(False, index=df.index)
    
    for col in features:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        col_outliers = (df[col] < lower) | (df[col] > upper)
        outlier_mask = outlier_mask | col_outliers
        
    print(f"[*] Detected outliers: {outlier_mask.sum()}")
    df = df[~outlier_mask].reset_index(drop=True)
    print(f"[+] Final cleaned dataset rows: {df.shape[0]} (Removed {orig_rows - df.shape[0]} rows)")

## 5. Splitting and Scaling

In [ ]:
if 'df' in locals():
    X = df[features]
    y = df['Flood']
    
    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    
    # Standard Scaling
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    print(f"Training set shape: {X_train_scaled.shape}")
    print(f"Testing set shape: {X_test_scaled.shape}")

## 6. Model Training & Comparison

In [ ]:
if 'df' in locals():
    models = {
        'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=8, class_weight='balanced'),
        'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, class_weight='balanced'),
        'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
        'XGBoost': XGBClassifier(random_state=42, eval_metric='logloss', scale_pos_weight=1.5)
    }
    
    results = []
    trained_clfs = {}
    
    for name, clf in models.items():
        clf.fit(X_train_scaled, y_train)
        trained_clfs[name] = clf
        y_pred = clf.predict(X_test_scaled)
        
        # Probas
        y_probs = clf.predict_proba(X_test_scaled)[:, 1] if hasattr(clf, "predict_proba") else y_pred
        
        acc = accuracy_score(y_test, y_pred)
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        auc = roc_auc_score(y_test, y_probs)
        
        results.append({
            'Model': name,
            'Accuracy': acc,
            'Precision': prec,
            'Recall': rec,
            'F1-Score': f1,
            'ROC-AUC': auc
        })
        
    results_df = pd.DataFrame(results)
    display(results_df)

In [ ]:
# Model Performance Plot
if 'df' in locals():
    melted = pd.melt(results_df, id_vars=['Model'], value_vars=['Accuracy', 'Precision', 'Recall', 'F1-Score'])
    plt.figure(figsize=(10, 5))
    sns.barplot(x='Model', y='value', hue='variable', data=melted, palette='Blues')
    plt.title('Performance Comparison of Models')
    plt.ylim(0, 1.1)
    plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left')
    plt.show()

In [ ]:
# ROC Curves
if 'df' in locals():
    plt.figure(figsize=(8, 6))
    for name, clf in trained_clfs.items():
        if hasattr(clf, "predict_proba"):
            probs = clf.predict_proba(X_test_scaled)[:, 1]
            fpr, tpr, _ = roc_curve(y_test, probs)
            auc = roc_auc_score(y_test, probs)
            plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], 'k--', label='Random (AUC = 0.50)')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curves')
    plt.legend(loc='lower right')
    plt.show()

## 7. Model Serialization
Select the best model (maximum F1-Score) and export.

In [ ]:
if 'df' in locals():
    best_idx = results_df['F1-Score'].idxmax()
    best_name = results_df.loc[best_idx, 'Model']
    best_clf = trained_clfs[best_name]
    
    print(f"Selected Best Model: {best_name}")
    
    # Export models
    os.makedirs('../models', exist_ok=True)
    joblib.dump(best_clf, '../models/flood_model.pkl')
    joblib.dump(scaler, '../models/scaler.pkl')
    print("[+] Serialized model and scaler successfully!")